# Doc2Vec で文書ベクトルを学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/06_word_vectors/doc2vec.ipynb)

NLP クックブック 06-11 ／ 単語ベクトル

文書そのものにベクトルを割り当てて、似た文書を探したい（word2vec の文書版）。

In [1]:
!pip install -q gensim

In [2]:
import nltk
nltk.download("reuters", quiet=True)
from nltk.corpus import reuters
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

ids = reuters.fileids()[:3000]
docs = [TaggedDocument([w.lower() for w in reuters.words(i) if w.isalpha()], [n]) for n, i in enumerate(ids)]
# workers=1 にすると、seed を固定したときに毎回同じ結果になる（スレッドが 2 つ以上だと処理の順番で結果が変わる）
model = Doc2Vec(docs, vector_size=100, window=5, min_count=3, epochs=20, dm=0, workers=1, seed=0)

In [3]:
q = 0
print("クエリ:", reuters.categories(ids[q]), " ".join(docs[q].words[:25]))
for n, sim in model.dv.most_similar(q, topn=3):
    print(f"{sim:.2f}", reuters.categories(ids[n]), " ".join(docs[n].words[:25]))

クエリ: ['trade'] asian exporters fear damage from u s japan rift mounting trade friction between the u s and japan has raised fears among many of asia
0.74 ['trade'] japan in last ditch effort to avert tariffs senior japanese officials tomorrow open talks with american trade negotiators in a last ditch effort to avert
0.74 ['trade'] shultz predicts rapid narrowing of u s trade gap secretary of state george shultz said the u s would erase its large foreign trade deficit
0.73 ['trade'] japan will not retaliate now against u s tariffs japan does not plan to take immediate retaliatory action against implementation of u s tariffs on


In [4]:
# 学習に使っていない新しい文書のベクトルを推論する
vec = model.infer_vector("oil prices rose after opec agreed to cut production".split(), epochs=50)
for n, sim in model.dv.most_similar([vec], topn=3):
    print(f"{sim:.2f}", reuters.categories(ids[n]), " ".join(docs[n].words[:15]))

0.79 ['crude'] malaysia advised to raise crude oil output in malaysia s national oil company petronas has
0.78 ['crude'] saudi arabia will not agree mad oil price rise saudi arabia s oil minister hisham
0.77 ['crude'] saudi arabia opposes drastic change in oil price saudi arabia s oil minister hisham nazer


- `dm=1` が PV-DM（文脈語 + 文書ベクトルで予測）、`dm=0` が PV-DBOW（文書ベクトルだけで単語を予測）。ここでは `dm=0` を使った。このデータでは `dm=1` にすると、上の推論の例で石油 (crude) の記事が上位に来なかった。
- 現在は事前学習済みの文埋め込みモデルのほうが手軽で高精度なことが多い。ドメイン特化の大量文書がある場合の選択肢。